# Level 3: NLP Sentiment Analysis

## Learning Objectives
- Preprocess and clean text data
- Apply NLP techniques (tokenization, stemming, lemmatization)
- Classify text into sentiment categories
- Analyze sentiment patterns and distributions
- Extract actionable insights from text data

## Step 1: Import Required Libraries and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('vader_lexicon', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# Setup visualization
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
images_dir = Path('../images')
images_dir.mkdir(exist_ok=True)

# Try to import wordcloud
try:
    from wordcloud import WordCloud
    has_wordcloud = True
except:
    has_wordcloud = False
    print('Note: WordCloud not installed')

print('All libraries imported successfully!')

## Step 2: Load and Explore Dataset

In [ ]:
# Load sentiment dataset
df = pd.read_csv('../datasets/3) Sentiment dataset.csv')

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Identify text and label columns
print("\nDataframe Info:")
print(df.info())

# Determine text column (usually contains longest strings)
for col in df.columns:
    if df[col].dtype == 'object':
        avg_length = df[col].astype(str).str.len().mean()
        print(f"{col}: avg length = {avg_length:.1f}")

In [ ]:
# Identify text column (longest average string)
text_col = None
max_avg_len = 0
for col in df.columns:
    if df[col].dtype == 'object':
        avg_len = df[col].astype(str).str.len().mean()
        if avg_len > max_avg_len:
            max_avg_len = avg_len
            text_col = col

# Identify label/sentiment column (likely has fewer unique values)
label_col = None
for col in df.columns:
    if col != text_col and df[col].dtype in ['object', 'int64']:
        unique_count = df[col].nunique()
        if unique_count <= 5:  # Assuming sentiment has few categories
            label_col = col
            break

print(f"Text column: {text_col}")
print(f"Label column: {label_col}")

if label_col:
    print(f"\nSentiment Distribution:")
    print(df[label_col].value_counts())

## Step 3: Text Preprocessing

In [ ]:
# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Clean and preprocess text"""
    if pd.isna(text):
        return ""
    
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def preprocess_text(text):
    """Full preprocessing with tokenization and lemmatization"""
    # Clean
    text = clean_text(text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens 
              if word not in stop_words and len(word) > 2]
    
    return ' '.join(tokens)

# Apply cleaning
print("Applying text preprocessing...")
df['text_cleaned'] = df[text_col].apply(clean_text)
df['text_processed'] = df[text_col].apply(preprocess_text)

print("\nOriginal vs Cleaned Text:")
for i in range(3):
    print(f"\nSample {i+1}:")
    print(f"Original: {df[text_col].iloc[i][:100]}...")
    print(f"Cleaned: {df['text_cleaned'].iloc[i][:100]}...")
    print(f"Processed: {df['text_processed'].iloc[i][:100]}...")

## Step 4: Sentiment Analysis - TextBlob

## Step 5: Sentiment Analysis - VADER

In [ ]:
# Apply VADER sentiment analysis
print("Analyzing sentiment with VADER...")

sia = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    scores = sia.polarity_scores(text)
    return scores['compound'], scores['pos'], scores['neu'], scores['neg']

vader_scores = df['text_cleaned'].apply(get_vader_sentiment)
df['vader_compound'] = vader_scores.apply(lambda x: x[0])
df['vader_pos'] = vader_scores.apply(lambda x: x[1])
df['vader_neu'] = vader_scores.apply(lambda x: x[2])
df['vader_neg'] = vader_scores.apply(lambda x: x[3])

# Classify based on VADER compound score
def classify_vader_sentiment(compound):
    if compound >= 0.05:
        return 'Positive'
    elif compound <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['vader_sentiment'] = df['vader_compound'].apply(classify_vader_sentiment)

print("\nVADER Sentiment Distribution:")
print(df['vader_sentiment'].value_counts())
print(f"\nCompound Score Statistics:")
print(df['vader_compound'].describe())

## Step 6: Sentiment Analysis Visualization

## Step 7: Text Statistics

# Visualize text statistics by sentiment
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sentiments = df['textblob_sentiment'].unique()
colors = {'Positive': '#2ecc71', 'Negative': '#e74c3c', 'Neutral': '#95a5a6'}
color_list = [colors.get(s, 'blue') for s in sentiments]

# Text length by sentiment
ax = axes[0]
df.boxplot(column='text_length', by='textblob_sentiment', ax=ax)
ax.set_title('Text Length by Sentiment')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Length (characters)')
plt.sca(ax)
plt.xticks(rotation=0)

# Word count by sentiment
ax = axes[1]
df.boxplot(column='word_count', by='textblob_sentiment', ax=ax)
ax.set_title('Word Count by Sentiment')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Word Count')
plt.sca(ax)
plt.xticks(rotation=0)

# Polarity by sentiment
ax = axes[2]
df.boxplot(column='polarity', by='textblob_sentiment', ax=ax)
ax.set_title('Polarity by Sentiment')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Polarity Score')
plt.sca(ax)
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('../images/17_text_statistics.png', dpi=100, bbox_inches='tight')
plt.show()
print('Text statistics visualization saved!')